# Synapse：放置、视图与参数

均一和异质突触使用同一套 API：均一值只是广播规则中的标量情况。这个例子只讨论一个 `Cell` population 上的突触声明、放置、查看和修改，不涉及外部事件或网络调度。

示例使用一个 10-CV、`pop_size=(4,)` 的真实多室 HH cell，并覆盖共享、矩形、ragged 和同位点重复放置。

In [1]:
import brainstate
import brainunit as u
import numpy as np
import pandas as pd

import braincell
from braincell import mech
from braincell.filter import AllRegion, LocsetMask, at

brainstate.environ.set(precision=64)

## 1. 构建 10-CV、4-cell population

morphology 包含一个 soma 和两条 dendrite。`CVPerBranchList((1, 4, 5))` 分别为三条 branch 分配 1、4、5 个 CV。四个 population 成员共享 morphology、CV 划分和 density mechanisms。

In [2]:
def build_morphology():
    soma = braincell.Branch.from_lengths(
        lengths=[20.0] * u.um,
        radii=[10.0, 10.0] * u.um,
        type="soma",
    )
    dend_a = braincell.Branch.from_lengths(
        lengths=[160.0] * u.um,
        radii=[2.0, 1.0] * u.um,
        type="basal_dendrite",
    )
    dend_b = braincell.Branch.from_lengths(
        lengths=[220.0] * u.um,
        radii=[2.5, 0.8] * u.um,
        type="apical_dendrite",
    )

    morphology = braincell.Morphology.from_root(soma, name="soma")
    morphology.soma.dend_a = dend_a
    morphology.soma.dend_b = dend_b
    return morphology

In [3]:
cell = braincell.Cell(
    build_morphology(),
    cv_policy=braincell.CVPerBranchList((1, 4, 5)),
    pop_size=(4,),
    V_init=-65.0 * u.mV,
    V_th=0.0 * u.mV,
    solver="staggered",
    name="postsynaptic_population",
)
cell.paint(
    AllRegion(),
    mech.CableProperty(
        resting_potential=-65.0 * u.mV,
        membrane_capacitance=1.0 * (u.uF / u.cm**2),
        axial_resistivity=100.0 * (u.ohm * u.cm),
    ),
    mech.Ion("SodiumFixed", E=50.0 * u.mV),
    mech.Ion("PotassiumFixed", E=-77.0 * u.mV),
    mech.Channel("Na_HH1952", g_max=120.0 * (u.mS / u.cm**2)),
    mech.Channel("K_HH1952", g_max=36.0 * (u.mS / u.cm**2)),
    mech.Channel("IL", g_max=0.3 * (u.mS / u.cm**2), E=-54.387 * u.mV),
)

assert cell.n_cv == 10
cv_table = pd.DataFrame({
    "cv_id": [cv.id for cv in cell.cvs],
    "branch_id": [cv.branch_id for cv in cell.cvs],
    "branch_type": [cv.branch_type for cv in cell.cvs],
    "prox": [cv.prox for cv in cell.cvs],
    "dist": [cv.dist for cv in cell.cvs],
    "midpoint": [(cv.prox + cv.dist) / 2 for cv in cell.cvs],
})
cv_table

,cv_id,branch_id,branch_type,prox,dist,midpoint
0,0,0,soma,0.00,1.00,0.500
1,1,1,basal_dendrite,0.00,0.25,0.125
2,2,1,basal_dendrite,0.25,0.50,0.375
3,3,1,basal_dendrite,0.50,0.75,0.625
4,4,1,basal_dendrite,0.75,1.00,0.875
5,5,2,apical_dendrite,0.00,0.20,0.100
6,6,2,apical_dendrite,0.20,0.40,0.300
7,7,2,apical_dendrite,0.40,0.60,0.500
8,8,2,apical_dendrite,0.60,0.80,0.700
9,9,2,apical_dendrite,0.80,1.00,0.900


## 2. 位置与参数广播

设选中的 cell 数为 `P`，矩形情况下每个 cell 有 `L` 个位置，ragged 情况第 `i` 个 cell 有 `L_i` 个位置，总实例数 `N = sum(L_i)`。`SynapseSpec(...)` 中的每个参数独立应用以下规则：

| 参数形状 | 含义 | 矩形 | ragged |
|---|---|---:|---:|
| scalar | 所有实例共享 | ✓ | ✓ |
| `(L,)` | 按 location 广播到每个 cell | ✓ | 仅当所有 `L_i` 相同 |
| `(P, 1)` | 每个 cell 一个值 | ✓ | ✓ |
| `(P, L)` | 每个 cell、每个 location 一个值 | ✓ | — |
| `(N,)` | 按最终 cell-major 实例顺序展开 | ✓ | ✓ |
| `P` 个逐行值（至少一行是 `(L_i,)`；其余行可为 scalar） | 逐 cell ragged 参数 | ✓ | ✓ |

后面的 `.set()` 已经作用在明确的 view 上，因此只接受 scalar 或 `(len(view),)`。

### 声明五组突触

`pf` 使用 scalar 参数；`aa` 演示 location 轴和 population 轴；`cf` 演示 ragged 展平参数。`co_exp` 与 `co_exp2` 用于后面的同位点多实例示例。

In [4]:
pf = mech.SynapseSpec(
    "ExpSyn", name="pf",
    tau=1.0 * u.ms, e=0.0 * u.mV,
)
aa = mech.SynapseSpec(
    "ExpSyn", name="aa",
    tau=np.asarray([1.2, 2.4]) * u.ms,          # (L,)
    e=np.asarray([[-5.0], [-20.0]]) * u.mV,    # (P, 1)
)
cf = mech.SynapseSpec(
    "Exp2Syn", name="cf",
    tau1=np.linspace(0.2, 0.7, 6) * u.ms,      # (N,), N = 1 + 2 + 0 + 3
    tau2=5.0 * u.ms,
    e=np.asarray([[-2.0], [-4.0], [-6.0], [-8.0]]) * u.mV,  # (P, 1)
)
co_exp = mech.SynapseSpec(
    "ExpSyn", name="co_exp",
    tau=1.5 * u.ms, e=-10.0 * u.mV,
)
co_exp2 = mech.SynapseSpec(
    "Exp2Syn", name="co_exp2",
    tau1=0.4 * u.ms, tau2=4.0 * u.ms, e=-10.0 * u.mV,
)

### 共享、矩形与 ragged 放置

`cell.place(...)` 把共享 locset 广播到完整 population。`cell[indices].place(...)` 只影响选中的成员。二维索引 `cell.cv_midpoints[...]` 返回与 population 行对齐的 `LocsetBatch`。

In [5]:
# 一个 pf 广播到全部四个 cell。
cell.place(at("dend_a", 0.25), pf)

# 两行分别对应 cell[1]、cell[3]，每行两个位置。
aa_locations = cell.cv_midpoints[np.asarray([[5, 8], [6, 9]])]
cell[[1, 3]].place(aa_locations, aa);

In [6]:
# 四个 cell 分别放置 [1, 2, 0, 3] 个 cf；空 LocsetMask 表示该行没有实例。
cf_locations = (
    at("dend_a", 0.50),
    at("dend_a", 0.35) | at("dend_b", 0.35),
    LocsetMask(),
    at("dend_a", 0.20) | at("dend_a", 0.80) | at("dend_b", 0.50),
)
cell.place(cf_locations, cf);

### 同位点仍是独立实例

`+` 连接 locset 时保留重复位置；`|` 表示去重并集。一次 `place(loc, syn1, syn2)` 会为每个 location 分别创建两个机制实例。所以下面得到四个 logical synapses，它们共享 electrical point，但不会合并。

In [7]:
duplicate_location = at("dend_b", 0.50) + at("dend_b", 0.50)
cell[2].place(duplicate_location, co_exp, co_exp2);

## 3. 使用 `SynapseView`

`cell.synapses` 是全部逻辑突触的根 view。view 不复制参数或状态，只保存稳定 logical ID 的选择。logical ID 不是表格行号，因此允许不连续。混合 type 的 view 可以读取公共 metadata；模型参数需要先选择为同一种 type。

In [8]:
def metadata_table(view):
    return pd.DataFrame({
        "id": view.id,
        "name": view.name,
        "synapse_type": view.synapse_type,
        "population_index": view.population_index,
        "branch_id": view.branch_id,
        "branch_x": view.branch_x,
        "cv_id": view.cv_id,
        "point_id": view.point_id,
    })

all_synapses = cell.synapses
all_instances = metadata_table(all_synapses)
assert len(all_synapses) == 18
all_instances

,id,name,synapse_type,population_index,branch_id,branch_x,cv_id,point_id
0,0,pf,ExpSyn,0,1,0.25,2,4
1,20,cf,Exp2Syn,0,1,0.50,3,5
2,1,pf,ExpSyn,1,1,0.25,2,4
3,5,aa,ExpSyn,1,2,0.10,5,8
4,9,aa,ExpSyn,1,2,0.70,8,11
5,25,cf,Exp2Syn,1,1,0.35,2,4
6,29,cf,Exp2Syn,1,2,0.35,6,9
7,2,pf,ExpSyn,2,1,0.25,2,4
8,46,co_exp,ExpSyn,2,2,0.50,7,10
9,50,co_exp2,Exp2Syn,2,2,0.50,7,10


In [9]:
coincident = cell.synapses[np.isin(cell.synapses.name, ["co_exp", "co_exp2"])]
assert len(coincident) == 4
assert len(np.unique(coincident.id)) == 4
assert len(np.unique(coincident.point_id)) == 1
metadata_table(coincident)

,id,name,synapse_type,population_index,branch_id,branch_x,cv_id,point_id
0,46,co_exp,ExpSyn,2,2,0.5,7,10
1,50,co_exp2,Exp2Syn,2,2,0.5,7,10
2,54,co_exp,ExpSyn,2,2,0.5,7,10
3,58,co_exp2,Exp2Syn,2,2,0.5,7,10


In [10]:
pf_by_name = cell.synapses["pf"]
pf_by_declaration = cell.synapses[pf]
exp_by_type = cell.synapses.by_type("ExpSyn")
cell_1_synapses = cell[1].synapses
selected_aa = cell[[1, 3]].synapses["aa"]
first_three = cell.synapses[:3]
one_instance = cell.synapses[0]

np.testing.assert_array_equal(pf_by_name.id, pf_by_declaration.id)
np.testing.assert_array_equal(pf_by_name.root.id, cell.synapses.id)
np.testing.assert_allclose(pf_by_name.get("tau").to_decimal(u.ms), [1.0] * 4)
assert [len(view) for view in (pf_by_name, exp_by_type, cell_1_synapses, selected_aa, first_three, one_instance)] == [4, 10, 5, 4, 3, 1]

pd.DataFrame({
    "selection": ["name='pf'", "type='ExpSyn'", "cell[1]", "cells[1,3]/aa", "[:3]", "[0]"],
    "count": [len(view) for view in (pf_by_name, exp_by_type, cell_1_synapses, selected_aa, first_three, one_instance)],
    "logical_ids": [view.id.tolist() for view in (pf_by_name, exp_by_type, cell_1_synapses, selected_aa, first_three, one_instance)],
})

,selection,count,logical_ids
0,name='pf',4,"[0, 1, 2, 3]"
1,type='ExpSyn',10,"[0, 1, 5, 9, 2, 46, 54, 3, 15, 19]"
2,cell[1],5,"[1, 5, 9, 25, 29]"
3,"cells[1,3]/aa",4,"[5, 9, 15, 19]"
4,[:3],3,"[0, 20, 1]"
5,[0],1,[0]


In [11]:
instance_counts = (
    all_instances.groupby(["population_index", "name"])
    .size()
    .unstack(fill_value=0)
    .reindex(
        index=range(4),
        columns=["pf", "aa", "cf", "co_exp", "co_exp2"],
        fill_value=0,
    )
)
np.testing.assert_array_equal(instance_counts.sum(axis=1), [2, 5, 5, 6])
instance_counts

name,pf,aa,cf,co_exp,co_exp2
population_index,,,,,
0,1,0,1,0,0
1,1,2,2,0,0
2,1,0,0,2,2
3,1,2,3,0,0


## 4. 初始化前修改参数

`.set()` 按当前 view 的顺序接受 scalar 或逐实例数组。声明中的矩形和 ragged 广播已经在 `aa`、`cf` 上展开；这里再把四个 `pf.tau` 修改为不同值。

In [12]:
pf_view = cell.synapses["pf"]
aa_view = cell.synapses["aa"]
cf_view = cell.synapses["cf"]

pf_view.set(tau=np.asarray([1.0, 1.5, 2.0, 2.5]) * u.ms)
cell.synapses["co_exp"].set(e=-12.0 * u.mV)

np.testing.assert_allclose(aa_view.tau.to_decimal(u.ms), [1.2, 2.4, 1.2, 2.4])
np.testing.assert_allclose(aa_view.e.to_decimal(u.mV), [-5.0, -5.0, -20.0, -20.0])
np.testing.assert_allclose(cf_view.tau1.to_decimal(u.ms), np.linspace(0.2, 0.7, 6))
np.testing.assert_allclose(cf_view.e.to_decimal(u.mV), [-2.0, -4.0, -4.0, -8.0, -8.0, -8.0])

exp_parameters = metadata_table(cell.synapses.by_type("ExpSyn"))
exp_parameters["tau_ms"] = cell.synapses.by_type("ExpSyn").tau.to_decimal(u.ms)
exp_parameters["e_mV"] = cell.synapses.by_type("ExpSyn").e.to_decimal(u.mV)
exp_parameters

,id,name,synapse_type,population_index,branch_id,branch_x,cv_id,point_id,tau_ms,e_mV
0,0,pf,ExpSyn,0,1,0.25,2,4,1.0,0.0
1,1,pf,ExpSyn,1,1,0.25,2,4,1.5,0.0
2,5,aa,ExpSyn,1,2,0.10,5,8,1.2,-5.0
3,9,aa,ExpSyn,1,2,0.70,8,11,2.4,-5.0
4,2,pf,ExpSyn,2,1,0.25,2,4,2.0,0.0
5,46,co_exp,ExpSyn,2,2,0.50,7,10,1.5,-12.0
6,54,co_exp,ExpSyn,2,2,0.50,7,10,1.5,-12.0
7,3,pf,ExpSyn,3,1,0.25,2,4,2.5,0.0
8,15,aa,ExpSyn,3,2,0.30,6,9,1.2,-20.0
9,19,aa,ExpSyn,3,2,0.90,9,12,2.4,-20.0


## 5. 初始化后的 parameter 与 state

| 操作 | 初始化前 | 初始化后 |
|---|---:|---:|
| `place(...)` | ✓ | —，结构已经固定 |
| `view.set(...)` | ✓ | ✓ |
| 读取/调用 `view.set_state(...)` | — | ✓ |

同一个 name 不能跨 synapse type 复用；混合 type 的 view 不能直接读取 type-specific 参数；参数 shape 或单位不兼容时会立即报错。

In [13]:
cell.init_state()

synapse_layouts = [layout for layout in cell.layouts if layout.kind.startswith("synapse:")]
layout_table = pd.DataFrame({
    "layout_id": [layout.id for layout in synapse_layouts],
    "kind": [layout.kind for layout in synapse_layouts],
    "n_active": [layout.n_active for layout in synapse_layouts],
})
layout_sizes = dict(zip(layout_table.kind, layout_table.n_active))
assert layout_sizes == {"synapse:ExpSyn": 10, "synapse:Exp2Syn": 8}
np.testing.assert_allclose(pf_view.tau.to_decimal(u.ms), [1.0, 1.5, 2.0, 2.5])
layout_table

,layout_id,kind,n_active
0,5,synapse:ExpSyn,10
1,6,synapse:Exp2Syn,8


In [14]:
# 只修改 pf view 的前两个参数；g 是动力学状态，使用 set_state()。
pf_view[:2].set(tau=np.asarray([3.0, 4.0]) * u.ms)
pf_view.set_state(g=np.asarray([0.10, 0.20, 0.30, 0.40]) * u.uS)

np.testing.assert_allclose(pf_view.get("tau").to_decimal(u.ms), [3.0, 4.0, 2.0, 2.5])
np.testing.assert_allclose(pf_view.get("g").to_decimal(u.uS), [0.10, 0.20, 0.30, 0.40])

post_init_pf = metadata_table(pf_view)
post_init_pf["runtime_index"] = pf_view.runtime_index
post_init_pf["tau_ms"] = pf_view.tau.to_decimal(u.ms)
post_init_pf["g_uS"] = pf_view.g.to_decimal(u.uS)
post_init_pf

,id,name,synapse_type,population_index,branch_id,branch_x,cv_id,point_id,runtime_index,tau_ms,g_uS
0,0,pf,ExpSyn,0,1,0.25,2,4,0,3.0,0.1
1,1,pf,ExpSyn,1,1,0.25,2,4,1,4.0,0.2
2,2,pf,ExpSyn,2,1,0.25,2,4,4,2.0,0.3
3,3,pf,ExpSyn,3,1,0.25,2,4,7,2.5,0.4


## 结论

- 均一和异质参数只是同一广播规则的不同输入形状。
- 共享、矩形和 ragged locations 最终都展开为 cell-major logical instances。
- 重复位置和一次放置多个 mechanisms 都保留独立实例，即使它们共享同一个 electrical point。
- `name` 用于 view 分组，`synapse_type` 决定 runtime layout。
- `.set()` 修改参数；初始化后的 `.set_state()` 修改动力学状态。
- `weight` 和 `delay` 属于 `Connection`，不属于 `SynapseSpec`；本 notebook 只展示未连接的突触实例。